# Intro

In [1]:
import os, sys
import numpy as np
import pickle 

In [2]:
from collections import defaultdict


In [3]:
from dcms.models import DCMModel, DECMModel, ADECMModel, DWCMModel

## Load data

In [4]:
HOME='/home/sarawalk/bowtie2_py39/bowtie2/'

In [5]:
sys.path.insert(0, HOME)

In [6]:
DATA_FOLDER=HOME+'dati_elezioni/'

In [7]:
files=os.listdir(DATA_FOLDER)
files.sort()

In [8]:
files

['crisi_dicos.csv',
 'crisi_weighted_edgelist.csv',
 'ita_elections_dicos.csv',
 'ita_elections_weighted_edgelist.csv',
 'quirinale_dicos.csv',
 'quirinale_weighted_edgelist.csv']

In [9]:
l_dataset=len(files)//2

## Functions

In [10]:
def el2ks(el):
    all_nodes=np.concatenate((el['source_id'], el['target_id']))
    all_nodes=np.unique(all_nodes)
    k_out=np.zeros(len(all_nodes), dtype=int)
    k_in=np.zeros(len(all_nodes), dtype=int)
    s_out=np.zeros(len(all_nodes), dtype=int)
    s_in=np.zeros(len(all_nodes), dtype=int)
    node_index={node:i for i,node in enumerate(all_nodes)}
    for s,t,w in el:
        i_s=node_index[s]
        i_t=node_index[t]
        k_out[i_s]+=1
        k_in[i_t]+=1
        s_out[i_s]+=w
        s_in[i_t]+=w
    return k_out, k_in, s_out, s_in, all_nodes

# Case0: Crisis

In [11]:
i_file=0

In [13]:
el=np.genfromtxt(
    DATA_FOLDER+files[i_file+1],
    delimiter=',',
    skip_header=1,
    autostrip=True,
    dtype=[('source_id', '>U50'), ('target_id', '>U20'),('weight', 'i4')]
 )

In [14]:
dico=np.genfromtxt(
    DATA_FOLDER+files[i_file],
    delimiter=',',
    skip_header=1,
    autostrip=True,
    dtype=[('user_id', '>U50'), ('dico', '>U2'), ('h_dico', 'U2'), ('i_dico', 'U2')]
 )

In [15]:
dico_dict={}
bad_dicos=[]
for d in dico:
    if d['dico'].isnumeric():
        dico_dict[d['user_id']]=int(d['dico'])
    else:
        if d['dico'] not in bad_dicos:
            bad_dicos.append(d['dico'])
            print(d['dico'])

-1


In [16]:
np.unique(list(dico_dict.values()), return_counts=True)

(array([0, 1, 2, 3, 4]), array([58832, 31874, 15168,  1304,  3637]))

In [21]:
n_nodes=np.concatenate((el['source_id'], el['target_id']))
n_nodes=np.unique(n_nodes)

In [22]:
len(n_nodes), len(dico_dict), len(dico_dict)/len(n_nodes)

(113876, 110815, 0.9731198847869612)

Who are the missing ones? I guees they are on smaller components

In [23]:
# Raggruppo in liste per efficienza, poi converto in array strutturati come el
_tmp = defaultdict(list)
for edge in el:
    src = edge['source_id'].strip()
    tgt = edge['target_id'].strip()
    d_src = dico_dict.get(src)
    if d_src is not None and d_src == dico_dict.get(tgt):
        _tmp[d_src].append(edge)

# defaultdict che restituisce array vuoti con lo stesso dtype di el
el_dico = defaultdict(
    lambda: np.empty(0, dtype=el.dtype),
    {k: np.array(v, dtype=el.dtype) for k, v in _tmp.items()}
 )

In [25]:
for key in el_dico.keys():
    print(key, len(el_dico[key]))

1 345876
0 322098
2 143937
3 1748
4 4578


In [26]:
del _tmp

## From the edgelist to the constraints

#### DiCo 1

In [27]:
aux=el2ks(el_dico[1])# DiCo 1

In [28]:
assert aux[0].sum()==aux[1].sum()==len(el_dico[1])

In [29]:
assert aux[2].sum()==aux[3].sum()

In [30]:
# Number of nodes, Number of edges, edge density
len(aux[4]), len(el_dico[1]), len(el_dico[1])/len(aux[4])**2

(31874, 345876, 0.0003404452594366783)

Ok, that's really challenging. So far, DCM Models were tested on datasets with 5k nodes. Here we have a little less than 32k nodes, and the number of links is a little more than 345k. Let's see if it works.

## DCM Models

### DWCM

In [38]:
# the "old" backend, i.e. pytorch
dwcm_old=DWCMModel(aux[2], aux[3])

In [ ]:
dwcm_old.solve_tool(tol=1e-5, max_iter=10000, backend='pytorch')

Converged in 27 iteration(s).


True

In [40]:
# with backend='pytorch'
with open('crisis_dwcm.pkl', 'wb') as f:
    pickle.dump(dwcm_old, f)

In [70]:
# the "old" backend, i.e. pytorch
dwcm_old2=DWCMModel(aux[2], aux[3])

In [71]:
dwcm_old2.solve_tool(tol=1e-5, max_iter=10000, backend='pytorch', variant='gauss-seidel')

RuntimeError: [enforce fail at alloc_cpu.cpp:118] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 130555904 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
# with backend='pytorch'
with open('crisis_dwcm4.pkl', 'wb') as f:
    pickle.dump(dwcm_old2, f)

In [31]:
dwcm=DWCMModel(aux[2], aux[3])

In [ ]:
dwcm.solve_tool(tol=1e-5, max_iter=10000)

Converged in 36 iteration(s).


True

In [35]:
# with backend='auto' (default), that is numba for N>5k
with open('crisis_dwcm2.pkl', 'wb') as f:
    pickle.dump(dwcm, f)

In [63]:
dwcm_gs=DWCMModel(aux[2], aux[3])

In [64]:
dwcm_gs.solve_tool(tol=1e-5, max_iter=10000, variant='gauss-seidel')

Converged in 80 iteration(s).


True

In [65]:
# with backend='auto' (default), that is numba for N>5k, but with GS
with open('crisis_dwcm3.pkl', 'wb') as f:
    pickle.dump(dwcm_gs, f)

#### Comparison

Numba, $\theta$-Newton

In [46]:
dwcm.max_relative_error(dwcm.sol.theta)

7.190263435667109e-10

In [51]:
dwcm.sol.elapsed_time//60

56.0

In [58]:
dwcm.sol.peak_ram_bytes

36736792

In [62]:
dwcm.sol.iterations

36

Numba, Gauss-Siedel

In [66]:
dwcm_gs.max_relative_error(dwcm_gs.sol.theta)

1.398220368835055e-10

In [67]:
dwcm_gs.sol.elapsed_time//60

90.0

In [68]:
dwcm_gs.sol.peak_ram_bytes

10404188

In [69]:
dwcm_gs.sol.iterations

80

PyTorch, $\theta$-Newton

In [47]:
dwcm_old.max_relative_error(dwcm_old.sol.theta)

1.2644781241810498e-09

In [52]:
dwcm_old.sol.elapsed_time//60

16.0

In [59]:
dwcm_old.sol.peak_ram_bytes

8416

In [61]:
dwcm_old.sol.iterations

27

PyTorch, Gauss-Siedel

In [72]:
dwcm_old2.max_relative_error(dwcm_old2.sol.theta)

AttributeError: 'NoneType' object has no attribute 'theta'

In [ ]:
dwcm_old2.sol.elapsed_time//60

16.0

In [ ]:
dwcm_old2.sol.peak_ram_bytes

8416

In [ ]:
dwcm_old2.sol.iterations

27

#### Summarising
It does not seems that there is any improvement in using numba...

### aDECM

In [34]:
adecm=ADECMModel(aux[0], aux[1], aux[2], aux[3])

In [146]:
adecm.solve_tool(tol=1e-5, max_iter=10000)

Topology: Converged in 19 iteration(s).


RuntimeError: [enforce fail at alloc_cpu.cpp:118] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 130555904 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
with open('crisis_adecm.pkl', 'wb') as f:
    pickle.dump(adecm, f)

### DECM

In [147]:
decm=DECMModel(aux[0], aux[1], aux[2], aux[3])

In [148]:
decm.solve_tool(tol=1e-5, max_iter=10000)

RuntimeError: [enforce fail at alloc_cpu.cpp:118] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 130555904 bytes. Error code 12 (Cannot allocate memory)

In [ ]:
with open('crisis_decm.pkl', 'wb') as f:
    pickle.dump(decm, f)